In [4]:
import pandas as pd
import pyarrow.feather as feather
import numpy as np
#feather_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step18.ftr"
# "please take the file: glm_data_step2_final.ftr and give it to Gary."

import os as os
#import matplotlib.pyplot as plt
import time as time
import random

#import modin.pandas as pd


In [3]:
"""1)	Open “superpolicy_final.sas7bdat” . a.	 This file has the superpolicy_id based on the logic Gary wrote in SAS i.	Where is that file (store for completion) 
2)	Read “necessary columns” from step 18 feather file since there will be memory issue otherwise. a.	“necessary columns” = columns needed for objective function, VIN, Date
b.	Merge this data with “superpolicy_final.sas7bdat” using VIN-Date 
    i.	Confirm the number of extra rows after the merge
    1.	Ideally there wont be any extra rows 
        i. Rows were not same. 
3)	Loop to find ideal fits run 500 simulation. 
    a.	For each loop get the 24D vector 
    b.	Calculate the Objective function for train, valid, holdout. 
    c.	If the error is less than SD, then proceed in the loop
    d.	Store at end of loop, seed, Objective function, time taken for each loop 
4)	Plot Loop vs Objective function. 
    Plot Hist of Objective function. 
    a.Choose lowest Objective function ? """




'1)\tOpen “superpolicy_final.sas7bdat” . a.\t This file has the superpolicy_id based on the logic Gary wrote in SAS i.\tWhere is that file (store for completion) \n2)\tRead “necessary columns” from step 18 feather file since there will be memory issue otherwise. a.\t“necessary columns” = columns needed for objective function, VIN, Date\nb.\tMerge this data with “superpolicy_final.sas7bdat” using VIN-Date \n    i.\tConfirm the number of extra rows after the merge\n    1.\tIdeally there wont be any extra rows \n        i. Rows were not same. \n3)\tLoop to find ideal fits run 500 simulation. \n    a.\tFor each loop get the 24D vector \n    b.\tCalculate the Objective function for train, valid, holdout. \n    c.\tIf the error is less than SD, then proceed in the loop\n    d.\tStore at end of loop, seed, Objective function, time taken for each loop \n4)\tPlot Loop vs Objective function. \n    Plot Hist of Objective function. \n    a.Choose lowest Objective function ? '

- Open “superpolicy_final.sas7bdat” . a.	 This file has the superpolicy_id based on the logic Gary wrote in SAS i.	Where is that file (store for completion) 

In [5]:
base_path = 'R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/'

full_feather_file_path = os.path.join(base_path,  "glm_data_step18.ftr")
columns_file_path = os.path.join(base_path,  "columns_glm_data_step2_final_feather.csv") 
superpolicy_file =  os.path.join(base_path, 'superpolicy_final.sas7bdat')

step18_ess_cols = os.path.join(base_path,  "step18_ess_cols.csv")
Seed66_train_file_path = os.path.join(base_path,  "Seed66_train_file_path.csv") 
Seed66_valid_file_path = os.path.join(base_path,  "Seed66_valid_file_path.csv") 
Seed66_holdout_file_path = os.path.join(base_path,  "Seed66_holdout_file_path.csv") 

""" 
feather_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.ftr"

csv_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"
csv_1000row_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"
csv_allrow_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.csv"


file_df1_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df1_1_superpolicy.csv"

file_df2_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df2_1_superpolicy.csv" """


' \nfeather_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.ftr"\n\ncsv_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"\ncsv_1000row_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"\ncsv_allrow_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.csv"\n\n\nfile_df1_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df1_1_superpolicy.csv"\n\nfile_df2_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df2_1_superpolicy.csv" '

In [5]:
df_a = pd.read_sas( superpolicy_file ,format = 'sas7bdat', encoding="ISO-8859-1")  # read sas7bdat file


In [6]:
#import modin.config as modin_cfg

#modin_cfg.Engine.put("ray")  # Modin will use Ray
#modin_cfg.Engine.put("dask")  # Modin will use Dask

#modin_cfg.Engine.put('unidist') # Modin will use Unidist
#unidist_cfg.Backend.put('mpi') # Unidist will use MPI backend

In [7]:
#df_a = pd.DataFrame(df_a_pandas)

In [8]:
df_a['Date'] = pd.to_datetime(df_a['Date'])
df_a['DateYMD'] = df_a['Date'].dt.strftime('%Y%m%d')
df_a['VIN_Date'] = df_a['VIN'] +'_' +  df_a['DateYMD']


In [9]:

df_a.drop(columns = ['DateYMD','VIN','Date','pol','ID'], inplace = True)

In [10]:
df_a = df_a.drop_duplicates( subset = 'VIN_Date', keep = 'first')

In [11]:
df_a.head()

,superpolicy_id,VIN_Date
0,1.0,1J4GK48K42W273250_20170101
1,2.0,1FMYU03122KA07434_20170101
2,2.0,1FMYU03122KA07434_20180101
3,4.0,2GTEC13V361342050_20170101
4,4.0,2GTEC13V361342050_20180103


In [12]:
#testing to make sure seed is repeatable
if 0:
    SeedNum = 1
    unique_values = df_a['superpolicy_id'].unique().tolist()
    np.random.seed(SeedNum)  
    shuffleunique_values = random.sample(unique_values, len(unique_values))
    temp = sum([a -b for a,b in zip(unique_values, shuffleunique_values) ])

    print(temp)
    unique_values = df_a['superpolicy_id'].unique().tolist()

#Experiment to confirm that random seed is reproducible. 
# Lesson: random seed and np.random seed are different things. 
# Since we are using random.sample since that is the only way to shuffle without altering original copy. 
# np.shuffle was changing original - that is , it was in place. 
if 0:
    for SeedNum in range(1,10):
        random.seed(SeedNum)  
        shuffleunique_values = random.sample(unique_values, len(unique_values))
        temp = sum(abs(np.array(unique_values)-  np.array(shuffleunique_values) ))
        print(temp)
    for SeedNum in range(1,10):
        random.seed(SeedNum)  
        shuffleunique_values = random.sample(unique_values, len(unique_values))
        temp = sum(abs(np.array(unique_values)-  np.array(shuffleunique_values) ))
        print(temp)

In [13]:
def get_tvh(df_a,SeedNum,sorted_unique_values,tv_split_colname):
    #all rows are assumed train
    # TrainPercentage = 0.6
    ValidPercentage = 0.2
    HoldoutPercentage = 0.2

    random.seed(SeedNum)  
    total_unique = len(unique_values)
    shuffleunique_values = random.sample(sorted_unique_values, total_unique)

    valid_end = int(total_unique * ValidPercentage)
    holdout_end = valid_end +  int(total_unique * HoldoutPercentage)
    

    #train_values = shuffleunique_values[:train_end]
    val_values = shuffleunique_values[:valid_end]
    holdout_values = shuffleunique_values[valid_end:holdout_end]


    value_to_split = {}
    for value in val_values:
        value_to_split[value] = 'valid'
    for value in holdout_values:
        value_to_split[value] = 'holdout'

    df_a[tv_split_colname] = df_a['superpolicy_id'].map(value_to_split)
    return df_a

In [14]:
def generate_row(df_subset):
    #np.random.seed(seed_num)
    
    agg_columns = ['ee_bi', 'ee_pd', 'ee_pip', 'ee_coll', 'ee_comp','ee_med',
       'incurred_raw_bi', 'incurred_raw_pd', 'incurred_raw_pip', 
       'incurred_raw_coll','incurred_raw_comp', 'incurred_raw_med'] 
    agg_dict = {col: 'sum' for col in agg_columns}
    df_subset_grouped = df_subset.groupby(['vc_Vehicle_Type']).agg(agg_dict).reset_index()

    #df_subset_grouped.head()
    df_subset_grouped['pp_bi'] = df_subset_grouped['incurred_raw_bi']/df_subset_grouped['ee_bi'] 
    df_subset_grouped['pp_pd'] = df_subset_grouped['incurred_raw_pd']/df_subset_grouped['ee_pd'] 
    df_subset_grouped['pp_pip'] = df_subset_grouped['incurred_raw_pip']/df_subset_grouped['ee_pip'] 
    df_subset_grouped['pp_med'] = df_subset_grouped['incurred_raw_med']/df_subset_grouped['ee_med'] 
    df_subset_grouped['pp_coll'] = df_subset_grouped['incurred_raw_coll']/df_subset_grouped['ee_coll'] 
    df_subset_grouped['pp_comp'] = df_subset_grouped['incurred_raw_comp']/df_subset_grouped['ee_comp'] 

    df_subset_grouped.set_index('vc_Vehicle_Type', inplace=True)
    flattened_columns = [f'{idx}_{col}' for idx in df_subset_grouped.index for col in df_subset_grouped.columns]
    flattened_data = df_subset_grouped.values.flatten()

    df_subset_flattened = pd.DataFrame([flattened_data], columns=flattened_columns)
    return df_subset_flattened

In [86]:
col_to_lead = ['vc_Gross_Combined_Weight']
test_df1 = pd.read_feather(full_feather_file_path, columns =  col_to_lead )




In [3]:
cols167_to_read = ['vc_Vehicle_Type_M',
'train',
'vc_MSRP',
'vc_Doors',
'vc_Acceleration_0_to_60',
'vc_Turning_Circle',
'vc_Length',
'vc_Wheelbase',
'vc_Bed_Length',
'vc_Width',
'vc_Height',
'vc_Ground_Clearance',
'vc_Curb_Weight',
'vc_Gross_Combined_Weight',
'vc_Interior_Volume',
'vc_Passenger_Volume',
'vc_Seating_Capacity',
'vc_Payload_Capacity',
'vc_Towing_Capacity',
'vc_Fuel_Tank_Capacity',
'vc_Seating_Rows',
'vc_Horsepower',
'vc_Torque',
'vc_Displacement',
'vc_Cylinders',
'vc_Engine_Valves',
'vc_drive_type_AWD',
'vc_drive_type_4WD',
'vc_drive_type_FWD',
'vc_drive_type_4X2',
'vc_drive_type_RWD',
'vc_engine_type_hybrid_electric',
'vc_aspiration_forced_induction',
'vc_fuel_diesel',
'vc_transmission_type_AUTOMATIC',
'vc_transmission_type_MANUAL',
'vc_transmission_type_CONTINUOUSLY_VARIABLE',
'vc_luxury_flag',
'vc_derived_volume_100k',
'vc_derived_density_100k',
'vc_derived_hp_to_volume',
'vc_derived_hp_to_density',
'vc_derived_hp_to_turn',
'vc_derived_hp_to_weight',
'vc_derived_turn_to_length',
'vc_derived_turn_to_width',
'vc_derived_length_to_rows',
'vc_derived_stability_val',
'vc_derived_info_screen_size',
'vc_Cargo_Van',
'vc_Active_Parking_Assistance',
'vc_Glass_Roof',
'vc_Power_Sunroof',
'vc_Adaptive_Cruise_Control',
'vc_Softtop',
'vc_Sunroof',
'vc_Mild_Hybrid',
'vc_Auto_Start_Stop',
'vc_Off_Road_Lights',
'vc_Variable_Power_Steering',
'vc_Auto_Dimming_Rearview_Mirror',
'vc_Auto_Dimming_Side_Mirrors',
'vc_Center_Limited_Slip_Differential',
'vc_Center_Locking_Differential',
'vc_Electronic_4WD_Selector',
'vc_Front_Limited_Slip_Differential',
'vc_Front_Locking_Differential',
'vc_Hill_Descent_Control',
'vc_Limited_Slip_Differential',
'vc_Paddle_Shifters',
'vc_Rear_Limited_Slip_Differential',
'vc_Rear_Locking_Differential',
'vc_DVD_Player',
'vc_Front_Parking_Sensors',
'vc_Heated_Steering_Wheel',
'vc_Headlight_Cleaners',
'vc_Rear_Window_Defogger',
'vc_Side_Window_Defogger',
'vc_Heated_Side_Mirrors',
'vc_Heated_Washer_Jets',
'vc_Tinted_Windows',
'vc_Integrated_WiFi',
'vc_Power_Folding_Side_Mirrors',
'vc_Rear_Wiper',
'vc_Rear_Wiper_Automatic',
'vc_Remote_Engine_Start',
'vc_Windshield_Heated',
'vc_Wipers_Speed_Sensitive',
'vc_Navigation_Touchscreen',
'vc_Parking_Sensors',
'vc_Rear_Parking_Sensors',
'vc_Semiautomatic_Parking_System',
'vc_Premium_Audio',
'vc_Premium_Wheels',
'vc_Touch_Screen',
'vc_Traffic_Sign_Recognition',
'vc_Satellite_Aided_Transmission',
'vc_Satellite_Radio',
'vc_Speed_Sensitive_Volume',
'vc_Subwoofer',
'vc_Surround_Sound',
'vc_Video_Display',
'vc_Video_System',
'vc_Wipers_Heated',
'vc_Active_Collision_Avoidance_System',
'vc_Audible_Forward_Collision_Warning',
'vc_Automatic_Emergency_Braking',
'vc_Automatic_Emergency_Steering',
'vc_Blind_Spot_Warning',
'vc_Cross_Traffic_Warning',
'vc_Forward_Automatic_Emergency_Braking',
'vc_Forward_Collision_Warning',
'vc_Front_Cross_Traffic_Warning',
'vc_Lane_Departure_Warning',
'vc_Lane_Keeping_Assistance',
'vc_Parking_Collision_Warning',
'vc_Passive_Collision_Avoidance_System',
'vc_Pedestrian_Detection',
'vc_Rear_Cross_Traffic_Warning',
'vc_Reverse_Automatic_Emergency_Braking',
'vc_Tactile_Forward_Collision_Warning',
'vc_Trailer_Assist',
'vc_Visual_Forward_Collision_Warning',
'vc_Active_Driving_Assistance',
'vc_Cornering_Brake_Control',
'vc_Driver_Attention_System',
'vc_Electronic_Braking_System',
'vc_Hill_Ascent_Assist',
'vc_Auto_Hazard_Flashers',
'vc_Automatic_Crash_Notification',
'vc_Crash_Sensing_Door_Locks',
'vc_Crash_Sensors',
'vc_Crumple_Zones',
'vc_Post_Crash_Fuel_Cutoff',
'vc_Precollision_System',
'vc_Antitheft_Key',
'vc_Vehicle_Immobilizer',
'vc_Automatic_High_Beams',
'vc_Backup_Camera',
'vc_Blind_Spot_Camera',
'vc_Camera',
'vc_Camera_Front',
'vc_Camera_Side',
'vc_Camera_Washer',
'vc_Cornering_Lights',
'vc_Daytime_Running_Lights',
'vc_Dual_Side_Mirrors',
'vc_Fog_Lights',
'vc_Fog_Lights_Rear',
'vc_Headlights_Auto_On_Off',
'vc_Headlights_Leveling',
'vc_Headlights_Wiper_Activated',
'vc_Night_Vision',
'vc_Side_Mirror_Turn_Signals',
'vc_Surround_View_Camera',
'vc_Taillights_LED',
'vc_Wipers_Automatic',
'ee_liab',
'rc_glm_pred_liab',
'incurred_cap_AME_VV_VT_liab',
'ee_coll',
'rc_glm_pred_coll',
'incurred_cap_AME_VV_VT_coll',
'ee_comp',
'rc_glm_pred_comp',
'incurred_cap_AME_VV_VT_comp',
    
]
df1_167 = pd.read_feather(full_feather_file_path, columns =  cols167_to_read )

In [83]:
temp = test_df1.value_counts()

In [85]:
temp.to_csv(os.path.join(base_path,  "vc_Gross_Combined_Weight.csv"))

In [15]:
#db_ppee  
#columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','claim_count_bi', 'claim_count_pd', 'claim_count_pip', 'claim_count_med','claim_count_coll', 'claim_count_comp']
columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp']
df1 = pd.read_feather(full_feather_file_path, columns =  columns_to_load )

#df_a = pd.DataFrame(df_a_pandas)
#write a function 
df1['Date'] = pd.to_datetime(df1['Date'])
df1['DateYMD'] = df1['Date'].dt.strftime('%Y%m%d')
df1['VIN_Date'] = df1['VIN'] +'_' +  df1['DateYMD']
df1.drop(columns = ['DateYMD','VIN','Date'], inplace = True)

In [16]:
csv_stat_file = 'df2_1_stat_pp_N500.csv'
stat_file = pd.read_csv(os.path.join(base_path, csv_stat_file))
stat_file['norm_mean_ee'] = stat_file['mean_ee']/np.sum(stat_file['mean_ee'])
stat_file.reset_index()
stat_file.head()

,Unnamed: 0,mean_pp,SDev,mean_ee,norm_mean_ee
0,CAR_pp_bi,147.732876,2.145207,2.504728e+06,0.094360
1,CAR_pp_pd,109.313637,0.624208,2.525973e+06,0.095160
2,CAR_pp_pip,94.588838,4.274305,7.547369e+05,0.028433
3,CAR_pp_med,29.121543,0.533489,1.844539e+06,0.069488
4,CAR_pp_coll,211.947742,0.984054,1.868875e+06,0.070405


In [17]:
def objectivefunction(train_df_a_pp,stat_file):
    mycol = train_df_a_pp.columns
    stat_file['sample'] = train_df_a_pp[mycol]
    objfunval_train_df = np.sum(np.abs(stat_file['sample'] - stat_file['mean_pp'])*stat_file['norm_mean_ee'])
    return objfunval_train_df


In [18]:
full_df_a = pd.merge(df_a, df1, on = 'VIN_Date', how = 'left')
full_df_a_subset_flattened = generate_row(full_df_a)

In [19]:
unique_values = df_a['superpolicy_id'].unique()
sorted_unique_values = sorted(unique_values)
tv_split_colname = 'tv_split' #+str(SeedNum)
pp_columns = stat_file['Unnamed: 0'] # [column for column in stat_file.columns if 'pp' in column]

listtotal_obj_fun  = []
metconditon = 1

incurred_columns = []
ee_columns = []
for i in range(len(pp_columns)):
    incurred_columns.append(pp_columns[i].replace('pp','incurred_raw'))
    ee_columns.append(pp_columns[i].replace('pp','ee'))


In [20]:
start_time = time.time()
listtotal_obj_fun = []
print(f"Seed,total_obj_fun,train_obj_fun, valid_obj_fun, holdout_obj_fun, elapsed (mins) ")
SeedNum = 0
tv_split_colname = 'tv_split'

for SeedNum in range(0,100):
    

    df_a[tv_split_colname] = 'train'
    train_df_a_subset_flattened = pd.DataFrame()
    valid_df_a = pd.DataFrame()
    holdout_df_a = pd.DataFrame()

    #get tv_split in df_a
    df_b = get_tvh(df_a,SeedNum,sorted_unique_values,tv_split_colname)

    # Get train, text, val 
    #train_df_a =  df_a.loc[df_a['tv_split'] == 'train'].reset_index().drop(['index','tv_split'], axis=1)
    valid_df_a =  df_b.loc[df_b['tv_split'] == 'valid'].reset_index().drop(['index','tv_split'], axis=1)
    holdout_df_a =  df_b.loc[df_b['tv_split'] == 'holdout'].reset_index().drop(['index','tv_split'], axis=1)

    valid_df_a = pd.merge(valid_df_a, df1, on = 'VIN_Date', how = 'left')
    holdout_df_a = pd.merge(holdout_df_a, df1, on = 'VIN_Date', how = 'left')

    valid_df_a_subset_flattened = generate_row(valid_df_a)
    holdout_df_a_subset_flattened = generate_row(holdout_df_a)

    #train_df_a_pp = generate_row_train( valid_df_a_subset_flattened,holdout_df_a_subset_flattened)
    for pp,inc,ee in zip(pp_columns,incurred_columns,ee_columns):
        #print(pp,inc,ee)
        train_df_a_subset_flattened[inc] = full_df_a_subset_flattened[inc] - holdout_df_a_subset_flattened[inc] -valid_df_a_subset_flattened[inc]
        train_df_a_subset_flattened[ee] = full_df_a_subset_flattened[ee] - holdout_df_a_subset_flattened[ee] - valid_df_a_subset_flattened[ee]
        train_df_a_subset_flattened[pp] = train_df_a_subset_flattened[inc]/train_df_a_subset_flattened[ee]

    valid_df_a_pp = valid_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    holdout_df_a_pp = holdout_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    train_df_a_pp = train_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)

    #send to objective function 

    train_objval = objectivefunction(train_df_a_pp,stat_file)
    valid_objval = objectivefunction(valid_df_a_pp,stat_file)
    holdout_objval = objectivefunction(holdout_df_a_pp,stat_file)

    #end_time = time.time()
    #elapsed_time = end_time - start_time
    #print(f"Elapsed time 3 : {elapsed_time : .2f} seconds")

    # train_objval = total_objval - valid_objval - holdout_objval 

    total_obj_fun = (train_objval + valid_objval + holdout_objval)/3
    #print(total_obj_fun)
    #print(f"Obj fn: {total_obj_fun : .2f, train_objval : .2f, valid_objval : .2f, holdout_objval : .2f} ")
    listtotal_obj_fun.append(total_obj_fun)  

    end_time = time.time()
    elapsed_time = end_time - start_time
    #print(f"Elapsed time 5 : {elapsed_time : .2f, total_obj_fun : .6f} seconds")

    print(f"{SeedNum:.2f},{total_obj_fun : .3f},{train_objval : .3f}, {valid_objval : .3f}, {holdout_objval : .3f} , {elapsed_time/60 : .2f}")


Seed,total_obj_fun,train_obj_fun, valid_obj_fun, holdout_obj_fun, elapsed (mins) 
0.00, 1.300, 1.040,  1.755,  1.105 ,  5.57
1.00, 1.343, 0.865,  1.506,  1.657 ,  10.73
2.00, 1.195, 0.841,  1.600,  1.143 ,  16.43
3.00, 1.404, 0.888,  1.879,  1.444 ,  22.03
4.00, 1.397, 0.739,  1.678,  1.775 ,  27.67
5.00, 1.210, 0.999,  1.527,  1.104 ,  33.22
6.00, 1.240, 1.040,  1.420,  1.260 ,  38.71
7.00, 1.303, 0.964,  1.524,  1.421 ,  43.74
8.00, 1.342, 1.157,  1.527,  1.342 ,  49.25
9.00, 1.215, 1.102,  1.400,  1.142 ,  54.19
10.00, 1.491, 1.066,  1.640,  1.768 ,  59.14
11.00, 1.340, 1.047,  1.302,  1.672 ,  64.39
12.00, 1.451, 0.951,  1.718,  1.685 ,  69.60
13.00, 1.320, 1.138,  1.294,  1.526 ,  74.58
14.00, 1.499, 0.913,  2.352,  1.232 ,  79.59
15.00, 1.104, 0.968,  1.161,  1.184 ,  84.62
16.00, 1.227, 0.897,  1.413,  1.371 ,  90.07
17.00, 1.250, 0.859,  1.298,  1.592 ,  95.03
18.00, 1.258, 1.208,  1.699,  0.867 ,  99.97
19.00, 1.345, 0.703,  1.753,  1.578 ,  104.90
20.00, 1.026, 0.923,  0.978,

KeyboardInterrupt: 

In [47]:
start_time = time.time()
listtotal_obj_fun = []
print(f"Seed,total_obj_fun,train_obj_fun, valid_obj_fun, holdout_obj_fun, elapsed (mins) ")
SeedNum = 0
tv_split_colname = 'tv_split'

for SeedNum in range(66,67):
    

    df_a[tv_split_colname] = 'train'
    train_df_a_subset_flattened = pd.DataFrame()
    valid_df_a = pd.DataFrame()
    holdout_df_a = pd.DataFrame()

    #get tv_split in df_a
    df_b = get_tvh(df_a,SeedNum,sorted_unique_values,tv_split_colname)

    # Get train, text, val 
    #train_df_a =  df_a.loc[df_a['tv_split'] == 'train'].reset_index().drop(['index','tv_split'], axis=1)
    valid_df_a =  df_b.loc[df_b['tv_split'] == 'valid'].reset_index().drop(['index','tv_split'], axis=1)
    holdout_df_a =  df_b.loc[df_b['tv_split'] == 'holdout'].reset_index().drop(['index','tv_split'], axis=1)

    valid_df_a = pd.merge(valid_df_a, df1, on = 'VIN_Date', how = 'left')
    holdout_df_a = pd.merge(holdout_df_a, df1, on = 'VIN_Date', how = 'left')

    valid_df_a_subset_flattened = generate_row(valid_df_a)
    holdout_df_a_subset_flattened = generate_row(holdout_df_a)

    #train_df_a_pp = generate_row_train( valid_df_a_subset_flattened,holdout_df_a_subset_flattened)
    for pp,inc,ee in zip(pp_columns,incurred_columns,ee_columns):
        #print(pp,inc,ee)
        train_df_a_subset_flattened[inc] = full_df_a_subset_flattened[inc] - holdout_df_a_subset_flattened[inc] -valid_df_a_subset_flattened[inc]
        train_df_a_subset_flattened[ee] = full_df_a_subset_flattened[ee] - holdout_df_a_subset_flattened[ee] - valid_df_a_subset_flattened[ee]
        train_df_a_subset_flattened[pp] = train_df_a_subset_flattened[inc]/train_df_a_subset_flattened[ee]

    valid_df_a_pp = valid_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    holdout_df_a_pp = holdout_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    train_df_a_pp = train_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)

    #send to objective function 

    train_objval = objectivefunction(train_df_a_pp,stat_file)
    valid_objval = objectivefunction(valid_df_a_pp,stat_file)
    holdout_objval = objectivefunction(holdout_df_a_pp,stat_file)

    #end_time = time.time()
    #elapsed_time = end_time - start_time
    #print(f"Elapsed time 3 : {elapsed_time : .2f} seconds")

    # train_objval = total_objval - valid_objval - holdout_objval 

    total_obj_fun = (train_objval + valid_objval + holdout_objval)/3
    #print(total_obj_fun)
    #print(f"Obj fn: {total_obj_fun : .2f, train_objval : .2f, valid_objval : .2f, holdout_objval : .2f} ")
    listtotal_obj_fun.append(total_obj_fun)  

    end_time = time.time()
    elapsed_time = end_time - start_time
    #print(f"Elapsed time 5 : {elapsed_time : .2f, total_obj_fun : .6f} seconds")

    print(f"{SeedNum:.2f},{total_obj_fun : .3f},{train_objval : .3f}, {valid_objval : .3f}, {holdout_objval : .3f} , {elapsed_time/60 : .2f}")

Seed,total_obj_fun,train_obj_fun, valid_obj_fun, holdout_obj_fun, elapsed (mins) 
66.00, 1.019, 1.181,  0.963,  0.915 ,  5.73


In [59]:
df_b = df_b.fillna('train')

In [63]:
train_df_a =  df_b.loc[df_b['tv_split'] == 'train'].reset_index().drop(['index','tv_split'], axis=1)

In [69]:
train_df_a = pd.merge(train_df_a, df1, on = 'VIN_Date', how = 'left')


In [75]:
train_df_a.head()

,superpolicy_id,VIN_Date,ee_bi,ee_pd,ee_pip,ee_med,ee_coll,ee_comp,vc_Vehicle_Type,incurred_raw_bi,incurred_raw_pd,incurred_raw_pip,incurred_raw_med,incurred_raw_coll,incurred_raw_comp
0,2.0,1FMYU03122KA07434_20170101,0.999,0.999,0.0,0.0,0.000,0.000,SUV,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,1FMYU03122KA07434_20180101,0.999,0.999,0.0,0.0,0.000,0.000,SUV,0.0,0.0,0.0,0.0,0.0,0.0
2,8.0,1FAHP3EN1BW189557_20170113,0.999,0.999,0.0,0.0,0.999,0.999,CAR,0.0,0.0,0.0,0.0,0.0,0.0
3,8.0,1FAHP3EN1BW189557_20180113,0.698,0.698,0.0,0.0,0.698,0.698,CAR,0.0,0.0,0.0,0.0,0.0,0.0
4,10.0,3N1AB6AP4CL662610_20170103,0.652,0.652,0.0,0.0,0.652,0.652,CAR,0.0,0.0,0.0,0.0,0.0,0.0


In [76]:
Seed66_train_file_path = os.path.join(base_path,  "Seed66_train_file_path.csv") 
Seed66_valid_file_path = os.path.join(base_path,  "Seed66_valid_file_path.csv") 
Seed66_holdout_file_path = os.path.join(base_path,  "Seed66_holdout_file_path.csv") 

train_df_a1 = train_df_a[['vc_Vehicle_Type','VIN_Date']]
valid_df_a1 = valid_df_a[['vc_Vehicle_Type','VIN_Date']]
holdout_df_a1 = holdout_df_a[['vc_Vehicle_Type','VIN_Date']]


train_df_a1.to_csv(Seed66_train_file_path,index = False)
valid_df_a1.to_csv(Seed66_valid_file_path,index = False)
holdout_df_a1.to_csv(Seed66_holdout_file_path,index = False)

In [42]:
train_df_a1['vc_Vehicle_Type'].value_counts()

vc_Vehicle_Type
TRUCK    4
SUV      1
Name: count, dtype: int64

In [30]:
base_path

'R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/'

In [ ]:
total_obj_fun = 0
train_objval = 0 
valid_objval = 0
holdout_objval = 0
elapsed_time = 0
print(f"Seed,total_obj_fun,train_obj_fun, valid_obj_fun, holdout_obj_fun, elapsed (mins) ")
print(f"{SeedNum:.2f},{total_obj_fun : .3f},{train_objval : .3f}, {valid_objval : .3f}, {holdout_objval : .3f} , {elapsed_time/60 : .2f}")



In [ ]:
total_obj_fun

In [ ]:
train_df_a_pp = train_df_a_subset_flattened[pp_columns].T
print(train_df_a_pp)

In [ ]:
valid_df_a_pp = valid_df_a_subset_flattened[pp_columns].T
print(valid_df_a_pp)

In [ ]:
holdout_df_a_pp = holdout_df_a_subset_flattened[pp_columns].T
print(holdout_df_a_pp)

In [ ]:
listtotal_obj_fun

In [ ]:
valid_df_a_subset_flattened[pp_columns].T

In [ ]:
 #merge with pp, ee db 
    train_df_a = pd.merge(train_df_a, df1, on = 'VIN_Date', how = 'left')
    train_df_a_subset_flattened = generate_row(train_df_a)
    train_df_a_pp = train_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    #send to objective function 
    train_objval = objectivefunction(train_df_a_pp,stat_file)


In [6]:
if 1:
    table = feather.read_table(full_feather_file_path)
    schema = table.schema
    col_names = schema.names

In [7]:
col_names

['index',
 'VIN',
 'Date',
 'ee_bi',
 'ee_pd',
 'ee_pip',
 'ee_med',
 'ee_coll',
 'ee_comp',
 'ee_um_uim',
 'ee_tot',
 'incurred_raw_bi',
 'incurred_raw_pd',
 'incurred_raw_pip',
 'incurred_raw_coll',
 'incurred_raw_comp',
 'incurred_raw_med',
 'incurred_raw_um_uim',
 'incurred_raw_tot',
 'claim_count_bi',
 'claim_count_pd',
 'claim_count_pip',
 'claim_count_coll',
 'claim_count_comp',
 'claim_count_med',
 'claim_count_um_uim',
 'claim_count_tot',
 'ep_bi',
 'ep_pd',
 'ep_pip',
 'ep_med',
 'ep_coll',
 'ep_comp',
 'ep_um_uim',
 'ep_tot',
 'incurred_cap_bi',
 'incurred_cap_pd',
 'incurred_cap_med',
 'incurred_cap_pip',
 'incurred_cap_um_uim',
 'incurred_cap_coll',
 'incurred_cap_comp',
 'incurred_cap_tot',
 'pol_py',
 'pol_late_payments',
 'pol_veh_count',
 'pol_driver_count',
 'veh_limit_person_bi',
 'veh_limit_pd',
 'veh_limit_med',
 'veh_limit_pip',
 'veh_limit_um',
 'veh_limit_uim',
 'veh_ded_coll',
 'veh_ded_comp',
 'driver_age',
 'eval_date',
 'driver_gender_male',
 'veh_use_commut

In [ ]:
#Read feather file columns
if 0:
    table = feather.read_table(full_feather_file_path)
    schema = table.schema
    col_names = schema.names
    df_columns = pd.DataFrame(col_names, columns = ['Column Names'])
    df_columns.to_csv(columns_file_path,index = False)
    df_columns.shape
    df_columns.nunique()
    duplicates = df_columns[df_columns.duplicated(keep= False)]
    print(duplicates)   


In [ ]:
#columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','ee_um_uim','ee_tot','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','incurred_raw_um_uim','incurred_raw_tot']
columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','claim_count_bi', 'claim_count_pd', 'claim_count_pip', 'claim_count_med','claim_count_coll', 'claim_count_comp']

In [ ]:
df1 = pd.read_feather(full_feather_file_path, columns =  columns_to_load )

In [ ]:
df1['Date'] = pd.to_datetime(df1['Date'])
df1['DateYMD'] = df1['Date'].dt.strftime('%Y%m%d')
df1['VIN_Date'] = df1['VIN'] +'_' +  df1['DateYMD']

In [ ]:
df1.shape

In [ ]:
df1.head()

In [ ]:
df1 = pd.merge(df1, df_a, on = 'VIN_Date', how = 'left')

In [ ]:
df1['VIN_Date'].nunique()

In [ ]:
df1.shape

In [ ]:
df_a['VIN_Date'].nunique()

In [ ]:
df1.shape

In [ ]:
def generate_row(df_subset):
    #np.random.seed(seed_num)
    
    agg_columns = ['ee_bi', 'ee_pd', 'ee_pip', 'ee_med', 'ee_coll', 'ee_comp',
       'claim_count_bi', 'claim_count_pd', 'claim_count_pip',
       'claim_count_coll', 'claim_count_comp', 'claim_count_med','incurred_raw_bi',
       'incurred_raw_pd', 'incurred_raw_pip', 'incurred_raw_coll',
       'incurred_raw_comp', 'incurred_raw_med'] 
    agg_dict = {col: 'sum' for col in agg_columns}
    df_subset_grouped = df_subset.groupby(['vc_Vehicle_Type']).agg(agg_dict).reset_index()

    #df_subset_grouped.head()
    df_subset_grouped['pp_bi'] = df_subset_grouped['incurred_raw_bi']/df_subset_grouped['ee_bi'] 
    df_subset_grouped['pp_pd'] = df_subset_grouped['incurred_raw_pd']/df_subset_grouped['ee_pd'] 
    df_subset_grouped['pp_pip'] = df_subset_grouped['incurred_raw_pip']/df_subset_grouped['ee_pip'] 
    df_subset_grouped['pp_med'] = df_subset_grouped['incurred_raw_med']/df_subset_grouped['ee_med'] 
    df_subset_grouped['pp_coll'] = df_subset_grouped['incurred_raw_coll']/df_subset_grouped['ee_coll'] 
    df_subset_grouped['pp_comp'] = df_subset_grouped['incurred_raw_comp']/df_subset_grouped['ee_comp'] 

    df_subset_grouped.set_index('vc_Vehicle_Type', inplace=True)
    flattened_columns = [f'{idx}_{col}' for idx in df_subset_grouped.index for col in df_subset_grouped.columns]
    flattened_data = df_subset_grouped.values.flatten()

    df_subset_flattened = pd.DataFrame([flattened_data], columns=flattened_columns)
    return df_subset_flattened

In [ ]:
df1_group_flattened = generate_row(df1)

In [ ]:
from itertools import chain 
ee_columns = [column for column in df1_group_flattened.columns if 'ee' in column]
pp_columns = [column for column in df1_group_flattened.columns if 'pp' in column]
all_columns = list(chain(pp_columns,ee_columns))
print(all_columns)

In [ ]:
csv_stat_file = 'df2_1_stat_pp_N500.csv'
stat_file = pd.read_csv(os.path.join(base_path, csv_stat_file))
stat_file.head()

In [ ]:
stat_file.index = stat_file['Unnamed: 0']

In [ ]:
stat_file.drop(columns = 'Unnamed: 0')

In [ ]:
df1_24D = df1_group_flattened[all_columns]

In [ ]:
df1_24D.head()

In [ ]:
df2_1_stat_pp_N500.csv

In [ ]:
df_d = pd.read_csv(os.path.join(fast_eda_path, 'df_design.csv'))

In [ ]:
np.max(df_a['superpolicy_id'])

In [ ]:
Num_unique = df_a['superpolicy_id'].nunique()


In [ ]:
unique_values = df_a['superpolicy_id'].unique()

In [ ]:


start_time = time.time()
#vSeedNum = 2#42
for SeedNum in range(1,6):
    tv_split_colname = 'tv_split_'+str(SeedNum)
    TrainPercentage = 0.6
    ValidPercentage = 0.2

    np.random.seed(SeedNum)  
    np.random.shuffle(unique_values)

    total_unique = len(unique_values)
    train_end = int(total_unique * TrainPercentage)
    val_end = train_end + int(total_unique * ValidPercentage)

    train_values = unique_values[:train_end]
    val_values = unique_values[train_end:val_end]
    holdout_values = unique_values[val_end:]

    value_to_split = {}
    for value in train_values:
        value_to_split[value] = 'train'
    for value in val_values:
        value_to_split[value] = 'valid'
    for value in holdout_values:
        value_to_split[value] = 'holdout'

    df_a[tv_split_colname] = df_a['superpolicy_id'].map(value_to_split)

    print(df_a[tv_split_colname].value_counts())

    end_time = time.time()
    elapsed_time = end_time - start_time

print(f"Elapsed time : {elapsed_time : .2f} seconds")


In [ ]:
df_a.shape

In [ ]:
df_a.columns

In [ ]:
df_a.drop('tv_split_5',axis = 1, inplace = True)

In [ ]:
df_a.columns

In [ ]:
df_a.head(10)

In [ ]:
if 0: 
    table = feather.read_table(feather_file_path)
    schema = table.schema
    col_names = schema.names
    df_columns = pd.DataFrame(col_names, columns = ['Column Names'])
    df_columns.to_csv(columns_file_path,index = False)

In [ ]:
# columns_to_load = ['ee_bi','ee_pd', 'ee_pip', 'ee_med','ee_coll','ee_comp','claim_count_bi','claim_count_pd','claim_count_pip','claim_count_coll','claim_count_comp','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','claim_count_med','VIN','Date','vc_Vehicle_Type','st']
columns_to_load = ['ee_bi','ee_pd', 'ee_pip', 'ee_med','ee_coll','ee_comp','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','VIN','Date','vc_Vehicle_Type']
df2 = pd.read_feather(full_feather_file_path , columns = columns_to_load) 

In [ ]:
df2.shape

In [ ]:
df2_1_superpolicy = pd.merge(df2,df_a, on = ['VIN','Date'], how = 'left')

In [ ]:
df2_1_superpolicy.shape

In [ ]:
(df2_1_superpolicy.shape[0] - df2.shape[0])/df2_1_superpolicy.shape[0]

In [ ]:
df2_1_superpolicy.head()

In [ ]:
df2_1_superpolicy.columns

In [ ]:

df2_1_superpolicy.to_csv(file_df2_1_superpolicy,index = False )

In [ ]:
df1_1_superpolicy = pd.merge(df1_1,df_a, on = ['ID'], how = 'inner')

In [ ]:
column_names = df1.columns.tolist()
columns_df = pd.DataFrame(column_names, columns = ['Column Names'] )
columns_df.to_csv(columns_file_path,index = False)

In [ ]:
df1['ID'] = np.arange(1,len(df1)+1)
df1.shape

In [ ]:
df1.columns

In [ ]:
df2 = pd.merge(df_a,df1,on = 'ID',how = 'left' )

In [ ]:
columns_to_write = ['ID','pol','VIN','Date',]

In [ ]:
df2 = df1[columns_to_write]

In [ ]:
df2.head()

In [ ]:
#df1.head(1000).to_csv(csv_1000row_file_path,index = False)

In [ ]:
df2.to_csv(csv_file_path, index = False)

In [ ]:
df2.to_csv(csv_allrow_file_path, index = False) 

In [ ]:
outfile = 'R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv'
df3 = pd.read_csv(outfile)

In [ ]:
df3.head()